# 15.7 - Evaluation Synthesis & Review

Status: VERIFIED

## What Are We Solving?
Knowing individual metrics is not enough. You must design an evaluation strategy for any system — from a simple classifier to a complex multi-agent pipeline.

## Mini Project: Complete Evaluation Pipeline

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    precision_recall_curve, f1_score, accuracy_score
)
import json
print("All imports OK")

All imports OK


## Step 1: Build Evaluation Dataset

In [2]:
# Create synthetic dataset with known properties
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    weights=[0.85, 0.15], random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

eval_dataset = {
    "version": "1.0",
    "total_samples": len(X_test),
    "class_distribution": {int(k): int(v) for k, v in zip(*np.unique(y_test, return_counts=True))},
}
print(f"Test set size: {eval_dataset['total_samples']}")
print(f"Class distribution: {eval_dataset['class_distribution']}")

Test set size: 400
Class distribution: {0: 339, 1: 61}


## Step 2: Train Multiple Models

In [3]:
models = {
    "LogisticRegression": LogisticRegression(random_state=42, max_iter=1000),
    "RandomForest_100": RandomForestClassifier(n_estimators=100, random_state=42),
    "RandomForest_200": RandomForestClassifier(n_estimators=200, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "f1": round(f1_score(y_test, y_pred), 4),
        "roc_auc": round(roc_auc_score(y_test, y_prob), 4),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    }
    print(f"{name}: Accuracy={results[name]['accuracy']}, F1={results[name]['f1']}, AUC={results[name]['roc_auc']}")

LogisticRegression: Accuracy=0.885, F1=0.5741, AUC=0.8917


RandomForest_100: Accuracy=0.9175, F1=0.6452, AUC=0.95


RandomForest_200: Accuracy=0.9175, F1=0.6452, AUC=0.9457


## Step 3: Error Analysis

In [4]:
# Analyze errors for best model
best_model_name = max(results, key=lambda k: results[k]["f1"])
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

# Find most uncertain samples
uncertainty = np.abs(y_prob_best - 0.5)
most_uncertain_idx = np.argsort(uncertainty)[:10]

print(f"Best model: {best_model_name}")
print(f"\nMost uncertain samples (sorted by uncertainty):")
print("Index | True | Prob   | Uncertainty")
print("-" * 45)
for idx in most_uncertain_idx:
    print(f"  {idx:3d} |  {y_test[idx]}   | {y_prob_best[idx]:.3f} | {uncertainty[idx]:.3f}")

# Error patterns
errors = y_pred_best != y_test
print(f"\nTotal errors: {errors.sum()} / {len(y_test)} ({errors.mean():.1%})")
print(f"False positives: {(y_pred_best == 1) & (y_test == 0).sum()}")
print(f"False negatives: {(y_pred_best == 0) & (y_test == 1).sum()}")

Best model: RandomForest_100

Most uncertain samples (sorted by uncertainty):
Index | True | Prob   | Uncertainty
---------------------------------------------
  335 |  1   | 0.490 | 0.010
  112 |  1   | 0.510 | 0.010
   57 |  1   | 0.520 | 0.020
   16 |  0   | 0.470 | 0.030
  332 |  0   | 0.470 | 0.030
  117 |  1   | 0.460 | 0.040
  262 |  1   | 0.460 | 0.040
  251 |  1   | 0.540 | 0.040
  380 |  0   | 0.450 | 0.050
  227 |  1   | 0.450 | 0.050

Total errors: 33 / 400 (8.2%)
False positives: [1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0
 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 1 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

## Step 4: Comprehensive Visualization

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Model comparison
model_names = list(results.keys())
f1_scores = [results[n]["f1"] for n in model_names]
axes[0, 0].bar(model_names, f1_scores, color=['steelblue', 'coral', 'seagreen'])
axes[0, 0].set_title('Model Comparison (F1)')
axes[0, 0].set_ylim(0, 1)
axes[0, 0].grid(True, alpha=0.3)

# Confusion matrix for best model
cm = np.array(results[best_model_name]["confusion_matrix"])
im = axes[0, 1].imshow(cm, cmap='Blues')
axes[0, 1].set_title(f'Confusion Matrix ({best_model_name})')
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_ylabel('True')
for i in range(2):
    for j in range(2):
        axes[0, 1].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=16)
plt.colorbar(im, ax=axes[0, 1])

# Precision-recall curve
precision_arr, recall_arr, _ = precision_recall_curve(y_test, y_prob_best)
axes[1, 0].plot(recall_arr, precision_arr, 'b-', linewidth=2)
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Precision-Recall Curve')
axes[1, 0].grid(True, alpha=0.3)

# Uncertainty distribution
axes[1, 1].hist(uncertainty, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Uncertainty (|prob - 0.5|)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Prediction Uncertainty Distribution')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('eval_synthesis.png', dpi=100, bbox_inches='tight')
plt.show()
print("Comprehensive evaluation saved")

Comprehensive evaluation saved


C:\Users\PC\AppData\Local\Temp\ipykernel_2696\1441161855.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 5: Write Evaluation Report

In [6]:
report = {
    "summary": {
        "best_model": best_model_name,
        "best_f1": results[best_model_name]["f1"],
        "best_accuracy": results[best_model_name]["accuracy"],
        "best_auc": results[best_model_name]["roc_auc"],
    },
    "model_comparison": results,
    "error_analysis": {
        "total_errors": int(errors.sum()),
        "false_positives": int(((y_pred_best == 1) & (y_test == 0)).sum()),
        "false_negatives": int(((y_pred_best == 0) & (y_test == 1)).sum()),
        "most_uncertain_indices": most_uncertain_idx.tolist(),
    },
    "recommendations": [
        "Consider threshold tuning to reduce false negatives",
        "Add more features to separate uncertain samples",
        "Monitor prediction distribution in production",
    ],
}

print("=== EVALUATION REPORT ===")
print(json.dumps(report["summary"], indent=2))
print(f"\nError Analysis:")
print(f"  Total errors: {report['error_analysis']['total_errors']}")
print(f"  False positives: {report['error_analysis']['false_positives']}")
print(f"  False negatives: {report['error_analysis']['false_negatives']}")
print(f"\nRecommendations:")
for r in report["recommendations"]:
    print(f"  - {r}")

=== EVALUATION REPORT ===
{
  "best_model": "RandomForest_100",
  "best_f1": 0.6452,
  "best_accuracy": 0.9175,
  "best_auc": 0.95
}

Error Analysis:
  Total errors: 33
  False positives: 2
  False negatives: 31

Recommendations:
  - Consider threshold tuning to reduce false negatives
  - Add more features to separate uncertain samples
  - Monitor prediction distribution in production


In [7]:
# Verification
assert best_model_name in results, "Must have best model"
assert results[best_model_name]["f1"] > 0.5, "F1 too low"
assert len(report["recommendations"]) > 0, "Must have recommendations"
print("VERIFICATION PASSED: Phase 15.7 complete")

VERIFICATION PASSED: Phase 15.7 complete
